# Cost-sensitive evaluation (baseline vs diffusion-augmented)

This notebook is where we connect the whole pipeline to the **real objective**:

> Minimize maintenance decision cost, where missing a real APS failure is much more expensive than a false alarm.

We evaluate:
1. A baseline model trained on the original training data
2. A model trained on a **diffusion-augmented** training dataset (synthetic failures added)

---

## Inputs

Expected (PCA space):
- `../../data/processed/pca_aps_mean_failure_train_set.csv`
- `../../data/processed/pca_aps_mean_failure_test_set.csv`

Optional (produced by diffusion notebook):
- `pca_aps_mean_failure_train_set_diffusion_augmented_<RUN_ID>.csv`

---

## Outputs

- Cost, confusion matrix, precision/recall/F1
- Threshold chosen to minimize cost on a validation split
- A comparison summary: baseline vs augmented


## 0) Imports + helper functions

In [ ]:
from pathlib import Path
import glob
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score
)
import matplotlib.pyplot as plt


### Define the cost function

From the APS case study, we treat:
- **False Positive (FP)**: unnecessary check (cost 10)
- **False Negative (FN)**: missed real failure (cost 500)

Cost = 10 * FP + 500 * FN


In [ ]:
CFP = 10   # cost of false positive
CFN = 500  # cost of false negative

def cost_from_confusion(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return CFP * fp + CFN * fn, {"tn": tn, "fp": fp, "fn": fn, "tp": tp}

def metrics_summary(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    cost, cm = cost_from_confusion(y_true, y_pred)
    return {
        "threshold": float(threshold),
        "cost": float(cost),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "roc_auc": float(roc_auc_score(y_true, y_prob)),
        "pr_auc": float(average_precision_score(y_true, y_prob)),
        **cm
    }


## 1) Load PCA train/test

If these files are missing, run the preprocessing notebook first:
- `01_DataExploration_Preprocessing_TUTORIAL.ipynb`


In [ ]:
train_path = Path("../../data/processed/pca_aps_mean_failure_train_set.csv")
test_path  = Path("../../data/processed/pca_aps_mean_failure_test_set.csv")

if not train_path.exists() or not test_path.exists():
    raise FileNotFoundError(
        "Missing PCA files. Expected ../../data/processed/pca_aps_mean_failure_train_set.csv and ../../data/processed/pca_aps_mean_failure_test_set.csv. "
        "Run the preprocessing notebook to generate them."
    )

train_df = pd.read_csv(train_path)
test_df  = pd.read_csv(test_path)

pc_cols = [c for c in train_df.columns if c.startswith("pc_")]
X = train_df[pc_cols].values
y = train_df["class"].values.astype(int)

X_test = test_df[pc_cols].values
y_test = test_df["class"].values.astype(int)

print("Train shape:", train_df.shape, "Test shape:", test_df.shape)
print("Train class counts:", pd.Series(y).value_counts().to_dict())
print("Test  class counts:", pd.Series(y_test).value_counts().to_dict())


## 2) Split train into train/validation

We choose the decision threshold using the validation split (to avoid tuning on the test set).


In [ ]:
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
(train_idx, val_idx), = sss.split(X, y)

X_train, y_train = X[train_idx], y[train_idx]
X_val, y_val     = X[val_idx], y[val_idx]

print("Train split:", X_train.shape, "Val split:", X_val.shape)
print("Val class counts:", pd.Series(y_val).value_counts().to_dict())


## 3) Train a baseline classifier

We start with Logistic Regression because it is:
- fast,
- interpretable,
- a strong baseline in many tabular problems.

We use `class_weight="balanced"` so the model pays more attention to failures.


In [ ]:
clf = LogisticRegression(max_iter=2000, class_weight="balanced")
clf.fit(X_train, y_train)

val_prob = clf.predict_proba(X_val)[:, 1]
test_prob = clf.predict_proba(X_test)[:, 1]


## 4) Choose a threshold that minimizes cost on validation

Accuracy is not the target. We pick the threshold that minimizes:

Cost = 10 * FP + 500 * FN


In [ ]:
thresholds = np.linspace(0.01, 0.99, 99)
costs = []
for th in thresholds:
    y_pred = (val_prob >= th).astype(int)
    cost, _ = cost_from_confusion(y_val, y_pred)
    costs.append(cost)

best_i = int(np.argmin(costs))
best_th = float(thresholds[best_i])
print("Best threshold on validation:", best_th, "with cost:", float(costs[best_i]))

plt.figure()
plt.plot(thresholds, costs)
plt.title("Validation cost vs threshold")
plt.xlabel("threshold")
plt.ylabel("cost")
plt.show()


## 5) Evaluate baseline model on test set

In [ ]:
baseline_test = metrics_summary(y_test, test_prob, threshold=best_th)
baseline_test


## 6) Evaluate diffusion-augmented training set (if available)

This cell looks for a file named like:

`pca_aps_mean_failure_train_set_diffusion_augmented_<RUN_ID>.csv`

If found, we retrain the same classifier and repeat the evaluation.


In [ ]:
aug_candidates = sorted(glob.glob("pca_aps_mean_failure_train_set_diffusion_augmented_*.csv"))
if not aug_candidates:
    print("No augmented dataset found. Run the diffusion notebook to generate one.")
else:
    aug_path = Path(aug_candidates[-1])  # take the most recent by filename sort
    print("Using augmented dataset:", aug_path)

    aug_df = pd.read_csv(aug_path)
    X_aug = aug_df[pc_cols].values
    y_aug = aug_df["class"].values.astype(int)

    # Re-split augmented data for fair threshold selection
    (train_idx2, val_idx2), = sss.split(X_aug, y_aug)
    X_train2, y_train2 = X_aug[train_idx2], y_aug[train_idx2]
    X_val2, y_val2     = X_aug[val_idx2], y_aug[val_idx2]

    clf2 = LogisticRegression(max_iter=2000, class_weight="balanced")
    clf2.fit(X_train2, y_train2)

    val_prob2 = clf2.predict_proba(X_val2)[:, 1]
    test_prob2 = clf2.predict_proba(X_test)[:, 1]

    thresholds = np.linspace(0.01, 0.99, 99)
    costs2 = []
    for th in thresholds:
        y_pred = (val_prob2 >= th).astype(int)
        cost, _ = cost_from_confusion(y_val2, y_pred)
        costs2.append(cost)

    best_i2 = int(np.argmin(costs2))
    best_th2 = float(thresholds[best_i2])
    print("Best threshold (augmented) on validation:", best_th2, "with cost:", float(costs2[best_i2]))

    plt.figure()
    plt.plot(thresholds, costs2)
    plt.title("Validation cost vs threshold (augmented)")
    plt.xlabel("threshold")
    plt.ylabel("cost")
    plt.show()

    augmented_test = metrics_summary(y_test, test_prob2, threshold=best_th2)
    augmented_test


## 7) Compare baseline vs augmented (if both exist)

The key comparison is the **test-set cost** and the FP/FN tradeoff.


In [ ]:
comparison = pd.DataFrame([baseline_test], index=["baseline"])

if "augmented_test" in globals():
    comparison = pd.concat([comparison, pd.DataFrame([augmented_test], index=["diffusion_augmented"])], axis=0)

comparison
